# Neural Networks in PySpark

Who has not had that dream? In the middle of the night, we lie awake and think of PySpark. Then, in the silvery moonshine, we think of neural networks and an ardent wish fills our hearts: how can we run deep learning models on Spark?

This notebook covers three things:
1. **PyTorch on Spark** — train a small CNN on MNIST using ``TorchDistributor``, Spark's mechanism for launching PyTorch jobs from a Spark session.
2. **Transfer learning** — use a pre-trained ResNet-50 to extract image features inside Spark via ``predict_batch_udf``, then classify them with both Spark ML and a PyTorch head trained through ``TorchDistributor``.
3. **Large-scale inference** — apply a pre-trained DistilBERT sentiment model to every line of Shakespeare, then use Spark SQL to analyse sentiment per play and per character.

### Setup

Install dependencies if running locally:

In [ ]:
!pip install torch torchvision pillow transformers pyarrow

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Deep_Learning")
    .config("spark.driver.memory", "12g")
    .getOrCreate()
)

### Shared utilities

``download_with_progress`` is used for the flower dataset.

In [ ]:
import urllib.request
import tarfile
from pathlib import Path

def download_with_progress(url: str, dest: Path) -> None:
    def _progress(count, block_size, total_size):
        pct = count * block_size / total_size * 100
        print(f"\r  Downloading... {min(pct, 100):.1f}%", end="", flush=True)
    print(f"Downloading from {url}")
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print(f"\n  Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

## PyTorch on Spark: MNIST

Before working with images and language, we introduce ``TorchDistributor`` on the simplest possible task: classifying handwritten digits.

The pattern is always the same:
1. Write a **self-contained training function** — all imports inside, no driver state.
2. Hand it to ``TorchDistributor.run()``.
3. Receive the trained ``state_dict`` back on the driver.

Everything else in the notebook follows this pattern.

In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyarrow.parquet as pq
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split


### The network

A small CNN: two convolutional layers to extract spatial features from the 28×28 images, max-pooling to downsample, then two linear layers down to 10 output logits — one per digit class. The forward pass returns raw logits; ``F.cross_entropy`` handles the softmax.

In [ ]:
# Tensor shapes throughout: B = batch size, C = channels, H = height, W = width
# e.g. [B, 1, 28, 28] means a batch of B grayscale images, each 28×28 pixels.

class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3),   # [B,  1, 28, 28] → [B, 32, 26, 26]  (32 filters, kernel 3×3)
            nn.ReLU(),
            nn.MaxPool2d(2),       # [B, 32, 26, 26] → [B, 32, 13, 13]  (halve spatial dims)
            nn.Conv2d(32, 64, 3),  # [B, 32, 13, 13] → [B, 64, 11, 11]
            nn.ReLU(),
            nn.Flatten(),          # [B, 64, 11, 11] → [B, 7744]         (64×11×11 = 7744)
            nn.Linear(7744, 128),
            nn.ReLU(),
            nn.Linear(128, 10),    # [B, 10] — one logit per digit class
        )

    def forward(self, x):
        return self.net(x)

### Let's code!

Fill in ``train_mnist`` below and run it via ``TorchDistributor``.

The function must be **fully self-contained** — all imports must be repeated inside the function body, because ``TorchDistributor`` pickles it and sends it to a separate worker process that does not share the driver's namespace.

Steps:
1. Repeat imports inside the function.
2. Download MNIST with ``datasets.MNIST`` and wrap in ``DataLoader``s.
3. Write a training loop for ``EPOCHS`` epochs using ``F.cross_entropy``.
4. Return ``model.state_dict()``.

See the ``TorchDistributor`` documentation for more details: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.torch.distributor.TorchDistributor.html

In [ ]:
EPOCHS     = 3
BATCH_SIZE = 64
LR         = 1e-3
DATA_PATH  = "/tmp/mnist"


def train_mnist():
    # All imports and class definitions must live inside the function:
    # TorchDistributor pickles it and sends it to a worker process that
    # does not share the driver's module namespace.
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, random_split

    # TODO: build a transforms.Compose pipeline with ToTensor and
    #       Normalize using MNIST mean/std (0.1307,) / (0.3081,).
    transform = ...

    # TODO: load the MNIST train and test sets with datasets.MNIST
    #       (use DATA_PATH, download=True, transform=transform).
    #       Then split the training set 80/20 into train_ds and val_ds
    #       with random_split.
    full_train = ...
    test_ds    = ...
    train_ds, val_ds = ...

    # TODO: wrap each of the three datasets in a DataLoader with BATCH_SIZE.
    #       Shuffle the training loader.
    train_loader = ...
    val_loader   = ...
    test_loader  = ...

    # TODO: pick a device (cuda if available, else cpu), instantiate
    #       MNISTNet on that device, and create an Adam optimizer with lr=LR.
    device = ...
    model  = ...
    opt    = ...

    # TODO: training loop for EPOCHS epochs.
    #       Each epoch: put the model in train() mode, iterate over train_loader,
    #       zero_grad, compute F.cross_entropy on model(images) vs labels,
    #       backward, step. Then evaluate on val_loader and print val accuracy.
    for epoch in range(1, EPOCHS + 1):
        ...

    # TODO: final evaluation on test_loader — report test accuracy.
    ...

    # TODO: return the trained model's state_dict so the driver can rebuild it.
    return ...


state_dict = TorchDistributor(
    num_processes=2, local_mode=True, use_gpu=False
).run(train_mnist)

# Reconstruct the model on the driver to enable local inference
trained_model = MNISTNet()
trained_model.load_state_dict(state_dict)
trained_model.eval()


### Downloading the flower dataset

We use the TensorFlow flower photos dataset — 3,670 images across 5 classes (daisy, dandelion, roses, sunflowers, tulips). The helper below downloads and extracts the archive with a progress indicator.

In [ ]:
URL      = "http://download.tensorflow.org/example_images/flower_photos.tgz"
DEST_DIR = Path("flower_photos")
ARCHIVE  = Path("flower_photos.tgz")

if not DEST_DIR.exists():
    if not ARCHIVE.exists():
        download_with_progress(URL, ARCHIVE)
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall()
else:
    print(f"{DEST_DIR}/ already exists, skipping download.")

### Loading images into Spark

Spark's ``binaryFile`` format reads image files as raw bytes into a DataFrame. Each row contains the file path and the raw content. We extract the flower class from the directory name (the second component of the path).

In [ ]:
from pyspark.sql.functions import col, split, element_at

images = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .option("recursiveFileLookup", "true")
    .load(str(DEST_DIR))
    .sample(fraction=0.1, seed=753)
)

# Path looks like: .../flower_photos/roses/123.jpg
# element_at with -2 picks the second-to-last segment (the class directory),
# which is robust regardless of how many leading path components there are.
images = images.withColumn(
    "label",
    element_at(split(col("path"), "/"), -2)
)

images.show(5, truncate=True)


### Extracting features with ResNet-50

We use **ResNet-50** with its default pretrained weights and remove the final classification head, so the model outputs a 2048-dimensional feature vector per image.

``predict_batch_udf`` takes a **setup function** — here ``make_resnet_fn`` — that each worker calls once at startup to load the model. The inner ``predict`` function it returns is then called repeatedly on batches of images.

In [ ]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import FloatType, ArrayType


def make_resnet_fn():
    """Called once per worker at startup: loads and caches the model."""
    import io, torch, numpy as np
    from PIL import Image
    from torchvision.models import resnet50, ResNet50_Weights
    from torchvision import transforms

    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model = torch.nn.Sequential(*list(model.children())[:-1])  # drop classifier head
    model.eval()

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],   # RGB channel means of ImageNet
                             std=[0.229, 0.224, 0.225]),    # RGB channel stds  of ImageNet
    ])

    def predict(image_bytes_batch: np.ndarray) -> np.ndarray:
        tensors = [
            preprocess(Image.open(io.BytesIO(b)).convert("RGB"))
            for b in image_bytes_batch
        ]
        with torch.no_grad():
            return model(torch.stack(tensors)).squeeze(-1).squeeze(-1).numpy()

    return predict


extract_features_udf = predict_batch_udf(
    make_resnet_fn,
    return_type=ArrayType(FloatType()),
    batch_size=32,
)

We repartition to control parallelism, then apply the UDF. Each partition is processed by one Python worker, which loads ResNet-50 once and runs it over its assigned images in batches of 32.

In [ ]:
# TODO: build features_df by selecting path, label, and the output of
#       extract_features_udf applied to the "content" column (aliased as "features").
#       Repartition to 8 first so each worker processes a roughly equal share,
#       and .cache() the result since we will reuse it below.
features_df = (
    images
    .repartition(8)
    .select(
        ...
    )
    .cache()
)

features_df.show(5, truncate=True)


### Exercise

Adapt the code below to run a multi-class flower classification using the ResNet features.

The features are stored as an ``ArrayType(FloatType())`` column. Use ``array_to_vector`` to convert them to the ``DenseVector`` format that Spark ML expects, then split into train/test and fit a classifier.

A ``LogisticRegression`` baseline is provided. Try swapping in a ``RandomForestClassifier`` or ``GradientBoostedTreesClassifier`` and compare accuracy.

In [ ]:
from pyspark.ml.functions import array_to_vector
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

indexer           = #... Use StringIndexer to generate the feature column
features_df_label = # Fit the indexer to the features dataset

# Use array_to_vector to convert the array to a vector for training
features_df_dense = features_df_label #...

# Perform train-test split
train, test = #...

Logistic regression in PySpark supports both binary and multinomial classification. For multinomial, the model estimates one probability per class and predicts the class with the highest score.

We add light L2 regularisation (``regParam``) to avoid overfitting on the small sample.

In [ ]:
# TODO: instantiate a classifier with featuresCol="features_vectorized"
#       and labelCol="labelIndex", then fit it on `train`.
#       A LogisticRegression with regParam=0.1 and maxIter=1000 is a good
#       starting point — try a RandomForestClassifier or
#       GBTClassifier as well and compare test accuracy below.
from pyspark.ml.classification import LogisticRegression  # plus others if you like

classifier = ...
logistic_model = ...


In [ ]:
# TODO: instantiate an evaluator for multiclass classification, fill in the correct parameters
# and evaluate your classification model from above
evaluator = #... 

### Training a classification head with TorchDistributor

The logistic regression baseline treats Spark ML as a black box. We can do better — and connect both halves of the notebook — by training a small PyTorch **classification head** on the same ResNet features using ``TorchDistributor``.

The flow is the same as in the Shakespeare section:
1. Spark writes the features to Parquet — the handoff point.
2. ``TorchDistributor`` launches PyTorch, which reads the Parquet files.
3. The trained weights come back to the driver as a ``state_dict``.

The model is deliberately simple: two fully-connected layers with ReLU and dropout. The ResNet has already done the hard visual work; the head only needs to learn which regions of the 2048-dimensional feature space correspond to each flower class.

In [ ]:
FLOWERS_TRAIN = "flowers_train.parquet"
FLOWERS_TEST  = "flowers_test.parquet"

# Persist the labelled feature vectors so TorchDistributor can read them
(
    train
    .select("features_vectorized", "labelIndex")
    .write.mode("overwrite").parquet(FLOWERS_TRAIN)
)
(
    test
    .select("features_vectorized", "labelIndex")
    .write.mode("overwrite").parquet(FLOWERS_TEST)
)

N_CLASSES      = int(features_df_dense.select("labelIndex").distinct().count())
FEATURE_DIM    = 2048
FLOWER_EPOCHS  = 20
FLOWER_BATCH   = 32
FLOWER_LR      = 1e-3

print(f"Classes: {N_CLASSES}, feature dim: {FEATURE_DIM}")
print(f"Train rows: {train.count()}, test rows: {test.count()}")

In [ ]:
class FlowerHead(nn.Module):
        """Two-layer classification head on top of frozen ResNet-50 features."""
        def __init__(self, in_dim, n_classes, dropout=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, n_classes),
            )

        def forward(self, x):
            return self.net(x)

In [ ]:
def train_flower_head():
    # All imports and class definitions must live inside the function
    # (same TorchDistributor worker-isolation requirement as train_mnist).
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    import pyarrow.parquet as pq
    from torch.utils.data import TensorDataset, DataLoader

    def load(path):
        """Read a Parquet file written by Spark and return a TensorDataset.

        Spark writes VectorUDT columns as structs {type, size, indices, values};
        to_pylist() therefore yields dicts — extract the 'values' key for the
        raw float list.
        """
        table = pq.read_table(path)
        X = torch.tensor(
            [row["values"] for row in table["features_vectorized"].to_pylist()],
            dtype=torch.float32,
        )
        y = torch.tensor(table["labelIndex"].to_pylist(), dtype=torch.long)
        return TensorDataset(X, y)

    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader = DataLoader(load(FLOWERS_TRAIN), batch_size=FLOWER_BATCH, shuffle=True)
    test_loader  = DataLoader(load(FLOWERS_TEST),  batch_size=FLOWER_BATCH)

    model = FlowerHead(FEATURE_DIM, N_CLASSES).to(device)
    opt   = optim.Adam(model.parameters(), lr=FLOWER_LR)

    for epoch in range(1, FLOWER_EPOCHS + 1):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            F.cross_entropy(model(X), y).backward()
            opt.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            total   += y.size(0)
    print(f"FlowerHead test accuracy: {correct / total:.4f}")

    return model.state_dict()


flower_state_dict = TorchDistributor(
    num_processes=1, local_mode=True, use_gpu=False,
).run(train_flower_head)

flower_model = FlowerHead(FEATURE_DIM, N_CLASSES)
flower_model.load_state_dict(flower_state_dict)
flower_model.eval()

The logistic regression and the PyTorch head are trained on the same features extracted by the same ResNet. Any accuracy difference is therefore purely due to the classifier — not the feature extractor. The PyTorch head has more capacity (non-linear activations, dropout regularisation) but also more hyperparameters to tune. On a dataset this small, the two are often competitive; the head's advantage grows with dataset size.

---

## Shakespeare Sentiment Analysis

We classify every Shakespeare line as positive or negative using a pre-trained DistilBERT model via HuggingFace Transformers. Inference runs inside Spark using `predict_batch_udf`, which loads the model once per worker and processes rows in batches — the same pattern used for ResNet feature extraction earlier in the notebook.

In [ ]:
from pyspark.sql.functions import avg, count, stddev, rand, expr, col, length
from pyspark.sql.types import FloatType, StringType
import pandas as pd

### Load and preprocess data

We read the CSV, keep only the three columns we need, and drop rows with missing lines. We also filter lines that are too short to convey a meaningful sentiment.

In [ ]:
df = spark.read.csv("shakespeare_data.csv", header=True, inferSchema=True)
df = df \
    .select("Play", "Player", "PlayerLine") \
    .na.drop(subset=["PlayerLine"]) \
    .filter(length(col("PlayerLine")) > 30)

### Exercise: build the sentiment UDF with predict_batch_udf

We use `predict_batch_udf` (the same API as the ResNet section) to run `distilbert-base-uncased-finetuned-sst-2-english` across all partitions. The model should be loaded once per worker at startup; the inner `predict` function receives a NumPy array of strings and returns a dict with two arrays — the predicted label and its confidence score.

Fill in the two stubs in the next cell: the inner `predict` body, and the `predict_batch_udf(...)` registration.


### Apply the UDF and extract sentiment class and confidence

The UDF returns a struct with two fields. We unpack them into separate columns so the rest of the pipeline can treat them as plain floats and strings.

In [ ]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType

# This part of the script relies on HuggingFace. If you have a HuggingFace account (which is not necessary
# to run the code), set an HF_TOKEN environment variable to accelerate the code.

def make_sentiment_fn():
    """Called once per worker: loads and caches the DistilBERT sentiment model."""
    from transformers import pipeline
    import numpy as np

    # TODO: build a HuggingFace sentiment-analysis pipeline using
    #       "distilbert-base-uncased-finetuned-sst-2-english".
    #       Set truncation=True and max_length=512 so long lines do not crash.
    classifier = ...

    def predict(texts: np.ndarray) -> dict:
        # TODO: run the classifier on texts (convert to a Python list first).
        #       Extract the "label" (lowercased) and "score" from each result
        #       into two numpy arrays, and return a dict with keys
        #       "sentiment_class" and "confidence".
        ...

    return predict


# TODO: register the UDF with predict_batch_udf. Pass make_sentiment_fn,
#       set return_type to the StructType below, and pick a batch_size
#       (256 is a reasonable default for short text on CPU).
sentiment_udf = ...

# Cache df and force materialization
# You may sample the dataframe (as done below) the accelerate the computation.
df = df.sample(fraction=0.25).cache()
df.count()

raw = df.repartition(4).withColumn("sentiment", sentiment_udf(col("PlayerLine")))
result = (
    raw
    .withColumn("sentiment_class", col("sentiment.sentiment_class"))
    .withColumn("confidence",      col("sentiment.confidence"))
    .drop("sentiment")
)


### Map classes to a numeric sentiment score

DistilBERT outputs binary labels (`positive` / `negative`). We map these to +1 / −1 so we can compute means and confidence intervals.

In [ ]:
from pyspark.sql.functions import when

# Binary mapping: positive → +1, negative → -1
result = result.withColumn(
    "sentiment_score",
    when(col("sentiment_class") == "positive",  1.0)
    .otherwise(-1.0)
    .cast(FloatType())
)

### Exercise: statistical analysis per play

Compute mean, standard deviation, line count, and a 95% confidence interval for each play's sentiment score.


In [ ]:
result.show(5)

In [ ]:
# TODO: group `result` by "Play" and aggregate:
#       - avg_sentiment    = mean of sentiment_score
#       - sentiment_stddev = stddev of sentiment_score
#       - line_count       = count of PlayerLine
#       - avg_confidence   = mean of confidence
#       Then add two columns for the 95% confidence interval of the mean:
#           ci_lower = avg_sentiment - 1.96 * sentiment_stddev / SQRT(line_count)
#           ci_upper = avg_sentiment + 1.96 * sentiment_stddev / SQRT(line_count)
#       Sort by avg_sentiment descending.
play_stats = (
    result
    .groupBy("Play")
    ...
)

print("Most positive plays:")
play_stats.show(5)

print("Most negative plays:")
play_stats.orderBy("avg_sentiment", ascending=True).show(5)


### Exercise: character-level sentiment

Which characters speak with consistently positive or negative dialogue? Aggregate per `Player`, then filter to the top/bottom 5 with at least 20 lines.


In [ ]:
# TODO: group `result` by "Player" and aggregate avg_sentiment,
#       line_count, and avg_confidence (same pattern as play_stats,
#       without the confidence interval). Sort by avg_sentiment descending.
character_stats = (
    result.groupBy("Player")
    ...
)

# TODO: filter character_stats to show the top 5 most positive characters
#       with at least 20 lines, then the top 5 most negative with the same
#       minimum line count. Hint: chain .filter(...) before .show(5).
print("Most positive characters (min 20 lines):")
...

print("Most negative characters (min 20 lines):")
...
